# Data audit

**Objective:** understand the competition datasets before training any model:
structure, quality, relationships between files, dates, geography, target
distribution and potential leakage.

**Input:** the raw competition files in `data/raw/`
(`Train.csv`, `Test.csv`, `climate_features.csv`, `SampleSubmission.csv`).

**Output:** `data/processed/train_merged.csv` and `data/processed/test_merged.csv`.

Main questions: dimensions, types, missing values, duplicates, class balance,
validity of the merge, temporal coverage, location overlap between Train and
Test, and suspicious or leaking variables.

## Imports

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (the notebook runs from notebooks/)
sys.path.insert(0, str(Path.cwd().parent))

import warnings

import matplotlib.pyplot as plt
import pandas as pd

from src.config import DATA_PROCESSED, DATA_RAW, TARGET

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## Load data

In [ ]:
train = pd.read_csv(DATA_RAW / "Train.csv")
test = pd.read_csv(DATA_RAW / "Test.csv")
climate = pd.read_csv(DATA_RAW / "climate_features.csv")
sample_submission = pd.read_csv(DATA_RAW / "SampleSubmission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Climate features shape:", climate.shape)
print("Sample submission shape:", sample_submission.shape)

## Columns

In [ ]:
print("TRAIN COLUMNS")
print(train.columns.tolist())

print("\nTEST COLUMNS")
print(test.columns.tolist())

print("\nCLIMATE COLUMNS")
print(climate.columns.tolist())

print("\nSAMPLE SUBMISSION COLUMNS")
print(sample_submission.columns.tolist())

## Data types

In [ ]:
for name, df in [("TRAIN", train), ("TEST", test), ("CLIMATE", climate)]:
    print(name)
    display(df.dtypes.to_frame("dtype"))

## Missing values

In [ ]:
def missing_report(df):
    """Missing count and percentage per column, sorted by count."""
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100,
    })
    return report.sort_values("missing_count", ascending=False)

for name, df in [("TRAIN", train), ("TEST", test), ("CLIMATE", climate)]:
    print(name)
    display(missing_report(df).head(10))

## Duplicates

In [ ]:
print("Duplicated rows")
print("  Train:", train.duplicated().sum())
print("  Test:", test.duplicated().sum())
print("  Climate:", climate.duplicated().sum())

print("\nDuplicated IDs")
print("  Train:", train["ID"].duplicated().sum())
print("  Test:", test["ID"].duplicated().sum())
print("  Climate:", climate["ID"].duplicated().sum())

## Target distribution

In [ ]:
print(train[TARGET].value_counts())
print("\nProportion:")
print(train[TARGET].value_counts(normalize=True))

train[TARGET].value_counts().sort_index().plot(kind="bar", figsize=(6, 4))
plt.title("Target Distribution")
plt.xlabel("Target")
plt.ylabel("Count")
plt.show()

## ID coverage

In [ ]:
print("Train IDs:", train["ID"].nunique())
print("Test IDs:", test["ID"].nunique())
print("Climate IDs:", climate["ID"].nunique())

# Parentheses matter: negate the boolean mask first, then count
print("\nTrain IDs missing from climate:", (~train["ID"].isin(climate["ID"])).sum())
print("Test IDs missing from climate:", (~test["ID"].isin(climate["ID"])).sum())

## Merge

In [ ]:
train_full = train.merge(
    climate, on="ID", how="left", validate="one_to_one", suffixes=("", "_climate")
)
test_full = test.merge(
    climate, on="ID", how="left", validate="one_to_one", suffixes=("", "_climate")
)

print("Merged train:", train_full.shape)
print("Merged test:", test_full.shape)

## Redundant column check

The merge creates `deathdate_climate`. If it is identical to `deathdate`,
it carries no information and must not be used as a feature.

In [ ]:
for name, df in [("train", train_full), ("test", test_full)]:
    if "deathdate_climate" in df.columns:
        same = (df["deathdate"].astype(str) == df["deathdate_climate"].astype(str)).mean()
        print(f"{name}: deathdate == deathdate_climate for {same:.1%} of rows")

## Dates

In [ ]:
for name, df in [("train", train_full), ("test", test_full)]:
    df["deathdate"] = pd.to_datetime(df["deathdate"], errors="coerce")
    print(f"\n{name.upper()}")
    print("Minimum date:", df["deathdate"].min())
    print("Maximum date:", df["deathdate"].max())

## Geographic distribution

In [ ]:
print("Train locations:", train_full["location"].nunique())
print("Test locations:", test_full["location"].nunique())

print("\nTrain location counts:")
display(train_full["location"].value_counts())

print("\nTest location counts:")
display(test_full["location"].value_counts())

## Train / Test location overlap

If most Test locations never appear in Train, a random validation split
overestimates real performance and a location-grouped split is required.

In [ ]:
train_locations = set(train_full["location"].dropna().unique())
test_locations = set(test_full["location"].dropna().unique())

print("Train locations:", len(train_locations))
print("Test locations:", len(test_locations))
print("Common locations:", len(train_locations & test_locations))
print(
    "Share of test rows in locations seen in train:",
    f"{test_full['location'].isin(train_locations).mean():.1%}",
)

print("\nLocations only in test:")
print(sorted(test_locations - train_locations))

## Climate variables

In [ ]:
climate_variables = [col for col in climate.columns if col != "ID"]
print("Number of climate variables:", len(climate_variables))
print(climate_variables)

## Save merged data

In [ ]:
train_full.to_csv(DATA_PROCESSED / "train_merged.csv", index=False)
test_full.to_csv(DATA_PROCESSED / "test_merged.csv", index=False)
print("Merged datasets saved.")